In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    multilabel_confusion_matrix,
    classification_report
)


In [60]:
#
# load final X and y arrays from 'final_X_and_y.npz'
#

from pathlib import Path
import pickle

directory_path_input = Path("D:/ECG/input_for_model") # chose the right path beforhand
loaded_arrays = np.load(directory_path_input/'X_y_input.npz', allow_pickle=True)

# chose the file to restore:
X_train = loaded_arrays['a']
y_train = loaded_arrays['b']
X_val = loaded_arrays['c']
y_val = loaded_arrays['d']
X_test = loaded_arrays['e']
y_test = loaded_arrays['f']
y_test
loaded_arrays.close()

In [62]:
# ============================================================
# 0) Configuration
# ============================================================

#SEED = 42
#SEED = 31
SEED = 7

tf.keras.utils.set_random_seed(SEED)

# Optional:
# This requests deterministic TensorFlow operations where supported.
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


OUTPUT_DIR = Path("cnn_multilabel_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#choose the right input_shape value for the input data used

model_config = {
    "input_shape": (1000, 1),
    "blocks": 4,
    "base_filters": 16,
    "kernel_size": 20,
    "dilation_mode": "exp",
    "dropout": 0.0,
    "head_units": 32,
    "use_batchnorm": True,
    "pooling_head": "gap",
    "kernel_initializer": "he_normal",
    "learning_rate": 0.5e-4,
    "batch_size": 64,
    "max_epochs": 300,
    "early_stopping_patience": 10,
    "random_seed": SEED,
    "num_labels": 4
}



In [64]:

# ============================================================
# 1) Prepare labels
# ============================================================

def prepare_multilabel_labels(y, num_labels=4):
    """
    Convert multi-label targets to a 2D float32 array.

    Expected target shape:
    - (N, num_labels)

    Expected values:
    - 0 or 1 for each label

    Example for 4 labels:
    [0, 1, 1, 0]
    """

    y = np.asarray(y)

    # Check dimensionality
    if y.ndim != 2:
        raise ValueError(
            "Multi-label targets must have shape (N, num_labels). "
            f"Received shape: {y.shape}"
        )

    # Check number of labels
    if y.shape[1] != num_labels:
        raise ValueError(
            f"Expected {num_labels} labels per sample, "
            f"but received shape {y.shape}."
        )

    # Convert to float32 for neural-network training
    y = y.astype(np.float32)

    # Check that only 0 and 1 occur
    unique_values = np.unique(y)

    if not np.all(np.isin(unique_values, [0.0, 1.0])):
        raise ValueError(
            "Multi-label targets must contain only 0 and 1. "
            f"Found values: {unique_values}"
        )

    return y


num_labels = model_config.get("num_labels", 4)

y_train_ml = prepare_multilabel_labels(
    y_train,
    num_labels=num_labels
)

y_val_ml = prepare_multilabel_labels(
    y_val,
    num_labels=num_labels
)

y_test_ml = prepare_multilabel_labels(
    y_test,
    num_labels=num_labels
)


# ------------------------------------------------------------
# Prepare X data
# ------------------------------------------------------------

X_train_model = np.asarray(X_train, dtype=np.float32)
X_val_model = np.asarray(X_val, dtype=np.float32)
X_test_model = np.asarray(X_test, dtype=np.float32)


# Add a channel dimension if data have shape (N, 1000).
if X_train_model.ndim == 2:
    X_train_model = X_train_model[..., np.newaxis]

if X_val_model.ndim == 2:
    X_val_model = X_val_model[..., np.newaxis]

if X_test_model.ndim == 2:
    X_test_model = X_test_model[..., np.newaxis]


# ------------------------------------------------------------
# Check X shapes
# ------------------------------------------------------------

if X_train_model.shape[1:] != model_config["input_shape"]:
    raise ValueError(
        f"X_train has model input shape {X_train_model.shape[1:]}, "
        f"but expected {model_config['input_shape']}."
    )

if X_val_model.shape[1:] != model_config["input_shape"]:
    raise ValueError(
        f"X_val has model input shape {X_val_model.shape[1:]}, "
        f"but expected {model_config['input_shape']}."
    )

if X_test_model.shape[1:] != model_config["input_shape"]:
    raise ValueError(
        f"X_test has model input shape {X_test_model.shape[1:]}, "
        f"but expected {model_config['input_shape']}."
    )


# ------------------------------------------------------------
# Check corresponding numbers of X and y samples
# ------------------------------------------------------------

if len(X_train_model) != len(y_train_ml):
    raise ValueError(
        "X_train and y_train contain different numbers of samples: "
        f"{len(X_train_model)} versus {len(y_train_ml)}."
    )

if len(X_val_model) != len(y_val_ml):
    raise ValueError(
        "X_val and y_val contain different numbers of samples: "
        f"{len(X_val_model)} versus {len(y_val_ml)}."
    )

if len(X_test_model) != len(y_test_ml):
    raise ValueError(
        "X_test and y_test contain different numbers of samples: "
        f"{len(X_test_model)} versus {len(y_test_ml)}."
    )


In [66]:
# ============================================================
# 2) Calculate multi-label weights
# ============================================================

label_names = ["SR", "ABQRS", "AFIB", "all_other"]

n_pos = np.sum(y_train_ml, axis=0)
n_neg = len(y_train_ml) - n_pos

if np.any(n_pos == 0):
    raise ValueError(
        "At least one label has no positive training samples."
    )

pos_weight = n_neg / n_pos

print("Positive label weights:")

for label, pos, neg, weight in zip(
    label_names,
    n_pos,
    n_neg,
    pos_weight
):
    print(
        f"{label:10s}: "
        f"positive={int(pos):5d}, "
        f"negative={int(neg):5d}, "
        f"pos_weight={weight:.3f}"
    )


Positive label weights:
SR        : positive= 6834, negative= 7544, pos_weight=1.104
ABQRS     : positive= 2254, negative=12124, pos_weight=5.379
AFIB      : positive= 1021, negative=13357, pos_weight=13.082
all_other : positive= 5495, negative= 8883, pos_weight=1.617


In [68]:

# ============================================================
# 3) CNN model builder
# ============================================================

def build_multilabel_cnn(
    input_shape=(1000, 1),
    blocks=4,
    base_filters=32,
    kernel_size=20,
    dilation_mode="exp",
    dropout=0.0,
    head_units=32,
    use_batchnorm=True,
    pooling_head="gap",
    kernel_initializer="he_normal",
    num_labels=4
):
    inputs = tf.keras.Input(
        shape=input_shape,
        name="ecg_input"
    )

    x = inputs

    for block_index in range(blocks):

        # Filter schedule:
        # block 1: 32 filters
        # block 2: 64 filters
        # block 3: 128 filters
        # block 4: 128 filters
        filters = base_filters * (2 ** min(block_index, 2))

        if dilation_mode == "exp":
            dilation_rate = 2 ** block_index
        elif dilation_mode == "none":
            dilation_rate = 1
        else:
            raise ValueError(
                "dilation_mode must be either 'none' or 'exp'."
            )

        x = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=kernel_size,
            dilation_rate=dilation_rate,
            padding="same",
            kernel_initializer=kernel_initializer,
            use_bias=not use_batchnorm,
            name=f"conv_block_{block_index + 1}"
        )(x)

        if use_batchnorm:
            x = tf.keras.layers.BatchNormalization(
                name=f"batchnorm_block_{block_index + 1}"
            )(x)

        x = tf.keras.layers.Activation(
            "relu",
            name=f"relu_block_{block_index + 1}"
        )(x)

        x = tf.keras.layers.MaxPooling1D(
            pool_size=2,
            name=f"maxpool_block_{block_index + 1}"
        )(x)

        if dropout > 0:
            x = tf.keras.layers.Dropout(
                rate=dropout,
                name=f"dropout_block_{block_index + 1}"
            )(x)

    if pooling_head == "gap":
        x = tf.keras.layers.GlobalAveragePooling1D(
            name="global_average_pooling"
        )(x)

    elif pooling_head == "flatten":
        x = tf.keras.layers.Flatten(
            name="flatten"
        )(x)

    else:
        raise ValueError(
            "pooling_head must be either 'gap' or 'flatten'."
        )

    if head_units > 0:
        x = tf.keras.layers.Dense(
            units=head_units,
            activation="relu",
            kernel_initializer=kernel_initializer,
            name="dense_head"
        )(x)

        if dropout > 0:
            x = tf.keras.layers.Dropout(
                rate=dropout,
                name="dense_head_dropout"
            )(x)

    outputs = tf.keras.layers.Dense(
        units=num_labels,
        activation="sigmoid",
        name="multilabel_output"
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="multilabel_ecg_cnn"
    )

    return model


model = build_multilabel_cnn(
    input_shape=model_config["input_shape"],
    blocks=model_config["blocks"],
    base_filters=model_config["base_filters"],
    kernel_size=model_config["kernel_size"],
    dilation_mode=model_config["dilation_mode"],
    dropout=model_config["dropout"],
    head_units=model_config["head_units"],
    use_batchnorm=model_config["use_batchnorm"],
    pooling_head=model_config["pooling_head"],
    kernel_initializer=model_config["kernel_initializer"]
)


In [70]:
# ============================================================
# 4) Weighted binary cross-entropy
# ============================================================

pos_weight_tf = tf.constant(
    pos_weight,
    dtype=tf.float32
)


def weighted_binary_crossentropy(y_true, y_pred):

    epsilon = tf.keras.backend.epsilon()

    y_pred = tf.clip_by_value(
        y_pred,
        epsilon,
        1.0 - epsilon
    )

    loss = -(
        pos_weight_tf
        * y_true
        * tf.math.log(y_pred)
        +
        (1.0 - y_true)
        * tf.math.log(1.0 - y_pred)
    )

    return tf.reduce_mean(
        loss,
        axis=-1
    )
    

In [72]:
# ============================================================
# 5) Compile model
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=model_config["learning_rate"]
    ),
    loss=weighted_binary_crossentropy,
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="binary_accuracy",
            threshold=0.5
        )
    ]
)


# ============================================================
# 6) Store architecture and model summary
# ============================================================

with open(
    OUTPUT_DIR / "model_configuration.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        model_config,
        file,
        indent=4,
        default=str
    )


with open(
    OUTPUT_DIR / "model_summary.txt",
    "w",
    encoding="utf-8"
) as file:
    model.summary(
        print_fn=lambda line: file.write(line + "\n")
    )


# Store the complete Keras model architecture as JSON.
with open(
    OUTPUT_DIR / "model_architecture.json",
    "w",
    encoding="utf-8"
) as file:
    file.write(model.to_json())



In [74]:
# ============================================================
# 7) Callbacks
# ============================================================

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=model_config["early_stopping_patience"],
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=OUTPUT_DIR / "best_multilabel_cnn.keras",
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        filename=OUTPUT_DIR / "training_log.csv",
        separator=",",
        append=False
    ),

    tf.keras.callbacks.TerminateOnNaN()
]


In [76]:
# ============================================================
# 8) Train model
# ============================================================

history = model.fit(
    X_train_model,
    y_train_ml,
    validation_data=(
        X_val_model,
        y_val_ml
    ),
    epochs=model_config["max_epochs"],
    batch_size=model_config["batch_size"],
    callbacks=callbacks,
    verbose=1,
    shuffle=True
)


# ============================================================
# 9) Store training history
# ============================================================

history_df = pd.DataFrame(history.history)
history_df.index = np.arange(1, len(history_df) + 1)
history_df.index.name = "epoch"

history_df.to_csv(
    OUTPUT_DIR / "training_history.csv"
)


# ============================================================
# 10) Generate class predictions
# ============================================================

train_probabilities = model.predict(
    X_train_model,
    batch_size=model_config["batch_size"],
    verbose=0
)

val_probabilities = model.predict(
    X_val_model,
    batch_size=model_config["batch_size"],
    verbose=0
)

test_probabilities = model.predict(
    X_test_model,
    batch_size=model_config["batch_size"],
    verbose=0
)


decision_threshold = 0.5

train_predictions = (
    train_probabilities >= decision_threshold
).astype(np.int32)

val_predictions = (
    val_probabilities >= decision_threshold
).astype(np.int32)

test_predictions = (
    test_probabilities >= decision_threshold
).astype(np.int32)

# ============================================================
# 11) Calculate final metrics
# ============================================================

# ------------------------------------------------------------
# Keras evaluation
# ------------------------------------------------------------

train_loss, train_binary_accuracy = model.evaluate(
    X_train_model,
    y_train_ml,
    batch_size=model_config["batch_size"],
    verbose=0
)

val_loss, val_binary_accuracy = model.evaluate(
    X_val_model,
    y_val_ml,
    batch_size=model_config["batch_size"],
    verbose=0
)

test_loss, test_binary_accuracy = model.evaluate(
    X_test_model,
    y_test_ml,
    batch_size=model_config["batch_size"],
    verbose=0
)


# ------------------------------------------------------------
# Exact-match / subset accuracy
# ------------------------------------------------------------

train_exact_accuracy = accuracy_score(
    y_train_ml,
    train_predictions
)

val_exact_accuracy = accuracy_score(
    y_val_ml,
    val_predictions
)

test_exact_accuracy = accuracy_score(
    y_test_ml,
    test_predictions
)


# ------------------------------------------------------------
# Macro F1 score
# ------------------------------------------------------------

train_f1_macro = f1_score(
    y_train_ml,
    train_predictions,
    average="macro",
    zero_division=0
)

val_f1_macro = f1_score(
    y_val_ml,
    val_predictions,
    average="macro",
    zero_division=0
)

test_f1_macro = f1_score(
    y_test_ml,
    test_predictions,
    average="macro",
    zero_division=0
)

# ------------------------------------------------------------
# Macro F1 score
# ------------------------------------------------------------

train_f1_micro = f1_score(
    y_train_ml,
    train_predictions,
    average="micro",
    zero_division=0
)

val_f1_micro = f1_score(
    y_val_ml,
    val_predictions,
    average="micro",
    zero_division=0
)

test_f1_micro = f1_score(
    y_test_ml,
    test_predictions,
    average="micro",
    zero_division=0
)


# ------------------------------------------------------------
# Per-label confusion matrices
# ------------------------------------------------------------

train_confusion_matrices = multilabel_confusion_matrix(
    y_train_ml,
    train_predictions
)

val_confusion_matrices = multilabel_confusion_matrix(
    y_val_ml,
    val_predictions
)

test_confusion_matrices = multilabel_confusion_matrix(
    y_test_ml,
    test_predictions
)


# ------------------------------------------------------------
# Per-class confusion matrices
# ------------------------------------------------------------

combination_to_class = {
    (1, 0, 0, 0): "SR",
    (0, 1, 0, 0): "ABQRS",
    (0, 0, 1, 0): "AFIB",
    (0, 0, 0, 1): "all_other_alone",
    (0, 1, 0, 1): "ABQRS + all_other",
    (0, 0, 1, 1): "AFIB + all_other",
    (0, 1, 1, 0): "ABQRS + AFIB",
    (0, 1, 1, 1): "ABQRS + AFIB + all_other"
}

y_train_classes = np.array([
    combination_to_class[tuple(row.astype(int))]
    for row in y_train_ml
])

y_val_classes = np.array([
    combination_to_class[tuple(row.astype(int))]
    for row in y_val_ml
])

y_test_classes = np.array([
    combination_to_class[tuple(row.astype(int))]
    for row in y_test_ml
])

y_train_pred_classes = np.array([
    combination_to_class.get(
        tuple(row.astype(int)),
        "invalid_combination"
    )
    for row in train_predictions
])

y_val_pred_classes = np.array([
    combination_to_class.get(
        tuple(row.astype(int)),
        "invalid_combination"
    )
    for row in val_predictions
])

y_test_pred_classes = np.array([
    combination_to_class.get(
        tuple(row.astype(int)),
        "invalid_combination"
    )
    for row in test_predictions
])

class_names = [
    "SR",
    "ABQRS",
    "AFIB",
    "all_other_alone",
    "ABQRS + all_other",
    "AFIB + all_other",
    "ABQRS + AFIB",
    "ABQRS + AFIB + all_other",
    "invalid_combination"
]

train_combination_confusion_matrix = confusion_matrix(
    y_train_classes,
    y_train_pred_classes,
    labels=class_names
)

test_combination_confusion_matrix = confusion_matrix(
    y_test_classes,
    y_test_pred_classes,
    labels=class_names
)

# ============================================================
# 12) Store final metrics
# ============================================================


final_metrics = {
    "num_labels": model_config["num_labels"],
    "number_of_model_parameters": int(model.count_params()),

    "decision_threshold": float(decision_threshold),

    "training": {
        "loss": float(train_loss),
        "binary_accuracy": float(train_binary_accuracy),
        "exact_match_accuracy": float(train_exact_accuracy),
        "macro_f1_score": float(train_f1_macro)
    },

    "validation": {
        "loss": float(val_loss),
        "binary_accuracy": float(val_binary_accuracy),
        "exact_match_accuracy": float(val_exact_accuracy),
        "macro_f1_score": float(val_f1_macro)
    },

    "test": {
        "loss": float(test_loss),
        "binary_accuracy": float(test_binary_accuracy),
        "exact_match_accuracy": float(test_exact_accuracy),
        "macro_f1_score": float(test_f1_macro)
    }
}


with open(
    OUTPUT_DIR / "final_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_metrics,
        file,
        indent=4
    )


# Also store the main metrics in table form.
metrics_df = pd.DataFrame(
    {
        "dataset": ["training", "validation", "test"],
        "loss": [
            train_loss,
            val_loss,
            test_loss
        ],
        "binary_accuracy": [
            train_binary_accuracy,
            val_binary_accuracy,
            test_binary_accuracy
        ],
        "exact_match_accuracy": [
            train_exact_accuracy,
            val_exact_accuracy,
            test_exact_accuracy
        ],
        "macro_f1_score": [
            train_f1_macro,
            val_f1_macro,
            test_f1_macro
        ]
    }
)

metrics_df.to_csv(
    OUTPUT_DIR / "final_metrics.csv",
    index=False
)
# ============================================================
# 12) Store confusion matrices
# ============================================================

# ------------------------------------------------------------
# 12a) Store label-wise 2x2 confusion matrices
# ------------------------------------------------------------

label_names = [
    "SR",
    "ABQRS",
    "AFIB",
    "all_other"
]

binary_confusion_index = [
    "true_negative",
    "true_positive"
]

binary_confusion_columns = [
    "predicted_negative",
    "predicted_positive"
]


for i, label in enumerate(label_names):

    train_label_cm_df = pd.DataFrame(
        train_confusion_matrices[i],
        index=binary_confusion_index,
        columns=binary_confusion_columns
    )

    test_label_cm_df = pd.DataFrame(
        test_confusion_matrices[i],
        index=binary_confusion_index,
        columns=binary_confusion_columns
    )

    train_label_cm_df.to_csv(
        OUTPUT_DIR / f"training_confusion_matrix_{label}.csv"
    )

    test_label_cm_df.to_csv(
        OUTPUT_DIR / f"test_confusion_matrix_{label}.csv"
    )


# ------------------------------------------------------------
# 12b) Store combination-class confusion matrices
# ------------------------------------------------------------



train_combination_cm_df = pd.DataFrame(
    train_combination_confusion_matrix,
    index=class_names,
    columns=class_names
)

test_combination_cm_df = pd.DataFrame(
    test_combination_confusion_matrix,
    index=class_names,
    columns=class_names
)


train_combination_cm_df.to_csv(
    OUTPUT_DIR / "training_combination_confusion_matrix.csv"
)

test_combination_cm_df.to_csv(
    OUTPUT_DIR / "test_combination_confusion_matrix.csv"
)
# ============================================================
# 13) Store detailed classification reports
# ============================================================

# ------------------------------------------------------------
# 13a) Multi-label classification reports
# ------------------------------------------------------------

label_names = [
    "SR",
    "ABQRS",
    "AFIB",
    "all_other"
]


train_report = classification_report(
    y_train_ml,
    train_predictions,
    target_names=label_names,
    output_dict=True,
    zero_division=0
)

val_report = classification_report(
    y_val_ml,
    val_predictions,
    target_names=label_names,
    output_dict=True,
    zero_division=0
)

test_report = classification_report(
    y_test_ml,
    test_predictions,
    target_names=label_names,
    output_dict=True,
    zero_division=0
)


train_report_df = pd.DataFrame(
    train_report
).transpose()

val_report_df = pd.DataFrame(
    val_report
).transpose()

test_report_df = pd.DataFrame(
    test_report
).transpose()


train_report_df.to_csv(
    OUTPUT_DIR / "training_multilabel_classification_report.csv"
)

val_report_df.to_csv(
    OUTPUT_DIR / "validation_multilabel_classification_report.csv"
)

test_report_df.to_csv(
    OUTPUT_DIR / "test_multilabel_classification_report.csv"
)

# ------------------------------------------------------------
# 12b) Combination-class classification reports
# ------------------------------------------------------------

train_combination_report = classification_report(
    y_train_classes,
    y_train_pred_classes,
    labels=class_names,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

val_combination_report = classification_report(
    y_val_classes,
    y_val_pred_classes,
    labels=class_names,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

test_combination_report = classification_report(
    y_test_classes,
    y_test_pred_classes,
    labels=class_names,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)


train_combination_report_df = pd.DataFrame(
    train_combination_report
).transpose()

val_combination_report_df = pd.DataFrame(
    val_combination_report
).transpose()


test_combination_report_df = pd.DataFrame(
    test_combination_report
).transpose()


train_combination_report_df.to_csv(
    OUTPUT_DIR / "training_combination_classification_report.csv"
)

val_combination_report_df.to_csv(
    OUTPUT_DIR / "validation_combination_classification_report.csv"
)

test_combination_report_df.to_csv(
    OUTPUT_DIR / "test_combination_classification_report.csv"
)

# ============================================================
# 14) Store predictions and probabilities
# ============================================================

label_names = [
    "SR",
    "ABQRS",
    "AFIB",
    "all_other"
]

train_prediction_data = {
    "true_combination_class": y_train_classes,
    "predicted_combination_class": y_train_pred_classes
}

val_prediction_data = {
    "true_combination_class": y_val_classes,
    "predicted_combination_class": y_val_pred_classes
}

test_prediction_data = {
    "true_combination_class": y_test_classes,
    "predicted_combination_class": y_test_pred_classes
}


for label_index, label_name in enumerate(label_names):

    # True labels
    train_prediction_data[f"true_{label_name}"] = (
        y_train_ml[:, label_index]
    )
    val_prediction_data[f"true_{label_name}"] = (
        y_val_ml[:, label_index]
    )
    test_prediction_data[f"true_{label_name}"] = (
        y_test_ml[:, label_index]
    )

    # Predicted labels
    train_prediction_data[f"predicted_{label_name}"] = (
        train_predictions[:, label_index]
    )
    val_prediction_data[f"predicted_{label_name}"] = (
        val_predictions[:, label_index]
    )
    test_prediction_data[f"predicted_{label_name}"] = (
        test_predictions[:, label_index]
    )

    # Predicted probabilities
    train_prediction_data[f"probability_{label_name}"] = (
        train_probabilities[:, label_index]
    )
    val_prediction_data[f"probability_{label_name}"] = (
        val_probabilities[:, label_index]
    )
    test_prediction_data[f"probability_{label_name}"] = (
        test_probabilities[:, label_index]
    )


train_prediction_df = pd.DataFrame(train_prediction_data)
val_prediction_df = pd.DataFrame(val_prediction_data)
test_prediction_df = pd.DataFrame(test_prediction_data)


train_prediction_df.to_csv(
    OUTPUT_DIR / "training_predictions.csv",
    index=False
)

val_prediction_df.to_csv(
    OUTPUT_DIR / "validation_predictions.csv",
    index=False
)

test_prediction_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)
# ============================================================
# 15) Save final model
# ============================================================

model.save(
    OUTPUT_DIR / "final_multi_label_cnn.keras"
)


# ============================================================
# 16) Print summary
# ============================================================

print("\nTraining completed.")
print(f"Results directory: {OUTPUT_DIR.resolve()}")
print(f"Number of parameters: {model.count_params():,}")
print(f"Decision threshold: {decision_threshold}")


# ------------------------------------------------------------
# Training results
# ------------------------------------------------------------

print("\nTraining metrics:")
print(f"Loss:                 {train_loss:.4f}")
print(f"Binary accuracy:      {train_binary_accuracy:.4f}")
print(f"Exact-match accuracy: {train_exact_accuracy:.4f}")
print(f"Macro F1-score:       {train_f1_macro:.4f}")

print("\nTraining label-wise confusion matrices:")
for label_name, cm in zip(
    label_names,
    train_confusion_matrices
):
    print(f"\n{label_name}:")
    print(cm)

print("\nTraining combination-class confusion matrix:")
print(train_combination_confusion_matrix)

print("\nTraining multi-label classification report:")
print(train_report_df)

print("\nTraining combination-class classification report:")
print(train_combination_report_df)


# ------------------------------------------------------------
# Validation results
# ------------------------------------------------------------

print("\nValidation metrics:")
print(f"Loss:                 {val_loss:.4f}")
print(f"Binary accuracy:      {val_binary_accuracy:.4f}")
print(f"Exact-match accuracy: {val_exact_accuracy:.4f}")
print(f"Macro F1-score:       {val_f1_macro:.4f}")

print("\nValidation multi-label classification report:")
print(val_report_df)

print("\nValidation combination-class classification report:")
print(val_combination_report_df)


# ------------------------------------------------------------
# Test results
# ------------------------------------------------------------

print("\nTest metrics:")
print(f"Loss:                 {test_loss:.4f}")
print(f"Binary accuracy:      {test_binary_accuracy:.4f}")
print(f"Exact-match accuracy: {test_exact_accuracy:.4f}")
print(f"Macro F1-score:       {test_f1_macro:.4f}")

print("\nTest label-wise confusion matrices:")
for label_name, cm in zip(
    label_names,
    test_confusion_matrices
):
    print(f"\n{label_name}:")
    print(cm)

print("\nTest combination-class confusion matrix:")
print(test_combination_confusion_matrix)

print("\nTest multi-label classification report:")
print(test_report_df)

print("\nTest combination-class classification report:")
print(test_combination_report_df)



Epoch 1/300
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step - binary_accuracy: 0.3663 - loss: 1.0296
Epoch 1: val_loss improved from None to 1.04888, saving model to cnn_multilabel_results\best_multilabel_cnn.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 49s 196ms/step - binary_accuracy: 0.4466 - loss: 0.9930 - val_binary_accuracy: 0.3420 - val_loss: 1.0489
Epoch 2/300
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step - binary_accuracy: 0.6153 - loss: 0.9132
Epoch 2: val_loss improved from 1.04888 to 1.03038, saving model to cnn_multilabel_results\best_multilabel_cnn.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 40s 177ms/step - binary_accuracy: 0.6361 - loss: 0.8946 - val_binary_accuracy: 0.3823 - val_loss: 1.0304
Epoch 3/300
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - binary_accuracy: 0.6801 - loss: 0.8385
Epoch 3: val_loss improved from 1.03038 to 0.89477, saving model to cnn_multilabel_results\best_multilabel_cnn.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 40s 177ms/step - binary_accuracy: 0.6901 - loss: 0.8246 - val_binar